# Who Has the Object? Response to Possession in Soccer and American Football

Analysis notebook for the invasion games paper. It puts SkillCorner (soccer) and Big Data Bowl (football) tracking on the same scale, metres and seconds, and tests five hypotheses:

| | Hypothesis | Rule behind it |
|---|---|---|
| **H1** | The gap between in-possession and out-of-possession shape is bigger in soccer | Central goal vs full-width end zone |
| **H2** | Soccer teams reshape within seconds of a turnover | Continuous vs stop-start play |
| **H3** | Soccer carriers keep less of their top speed, especially through turns | Feet vs hands (shared limb) |
| **H4** | Possession changes hands far more often in soccer, and more progress comes by pass | Turnovers and value per moment |
| **H5** | Football players cover less distance per live minute but sprint more of it | Rest between plays |

**Data (both free, both from video tracking)**
1. **Soccer:** SkillCorner open data, A-League 2024/25 broadcast tracking at 10 fps. Clone with Git LFS, otherwise the tracking files are tiny placeholders:
   `git lfs install && git clone https://github.com/SkillCorner/opendata.git data/skillcorner`
2. **Football:** StatsBomb American Football open data (#TB12DB). All 22 players on every scrimmage play at 30 fps, for every Tom Brady game from 2016 to 2022, both teams' offences. The ball isn't tracked, but StatsBomb's events tag the ball carrier every 0.1 s:
   `git clone https://github.com/statsbomb/amf-open-data.git data/statsbomb-amf`
3. **Play-by-play:** nflverse, downloaded automatically and filtered to the same games.

Attribution: StatsBomb asks that published work credits StatsBomb as the source and uses its logo (see the repo's licence).

In [10]:
# !pip install numpy pandas scipy matplotlib pyarrow tabulate

## 1. Settings
Set your paths here. Keep the `MAX_*` limits for a quick first run, then set them to `None` for the full data.

In [11]:
from pathlib import Path

DATA = Path("data")
SKILLCORNER_DIR = DATA / "skillcorner"      # SkillCorner/opendata clone
STATSBOMB_DIR = DATA / "statsbomb-amf"      # statsbomb/amf-open-data clone
OUT = Path("results")

MAX_MATCHES = 2      # soccer matches (None = all 20)
MAX_GAMES = 4        # football games (None = all ~124)
SOCCER_STRIDE = 2    # team shape every Nth frame (both sports run at 10 fps)
FOOTBALL_STRIDE = 2

OUT.mkdir(parents=True, exist_ok=True)

In [12]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull, QhullError

pd.set_option("display.max_columns", 30)
import gzip
%matplotlib inline

### Check the data is in place

In [13]:
tracking = sorted((SKILLCORNER_DIR / "data" / "matches").glob("*/*_tracking_extrapolated.jsonl"))
pointers = [p for p in tracking if p.stat().st_size < 1000]
print(f"SkillCorner: {len(tracking)} matches" + (f" -- {len(pointers)} are Git LFS placeholders, run `git lfs pull`" if pointers else ""))
games = sorted((STATSBOMB_DIR / "data" / "tracking").glob("*.json.gz"))
events = sorted((STATSBOMB_DIR / "data" / "events").glob("*.csv"))
print(f"StatsBomb: {len(games)} tracking games, {len(events)} event files")

SkillCorner: 20 matches
StatsBomb: 124 tracking games, 6 event files


## 2. Shared metrics
Sport-agnostic metrics: team shape (width, length, spread, convex hull), speed and turn rate, sprint profile, and speed with vs without the object. Thresholds are at the top.

In [ ]:
HSR_MS = 5.5        # high-speed running, ~19.8 km/h
SPRINT_MS = 7.0     # sprinting, ~25.2 km/h (a common soccer threshold)
REL_SPRINT = 0.85   # sprint relative to a player's own top speed
MIN_TURN_SPEED = 1.5  # m/s; below this, heading is too noisy to measure turns


def hull_area(xy: np.ndarray) -> float:
    xy = xy[~np.isnan(xy).any(axis=1)]
    if len(xy) < 3:
        return np.nan
    try:
        return ConvexHull(xy).volume  # in 2D, .volume is the area
    except QhullError:
        return np.nan


def team_shapes(df: pd.DataFrame, keys: list, x: str = "x", y: str = "y") -> pd.DataFrame:
    """One row per group (e.g. frame + team): width, length, spread, convex hull area, centroid.

    Any column that is constant within a group (team state, own goal x...) can be
    added to `keys` so it is carried through to the output.
    """
    g = df.groupby(keys, sort=False)
    out = g.agg(cx=(x, "mean"), cy=(y, "mean"), xmin=(x, "min"), xmax=(x, "max"),
                ymin=(y, "min"), ymax=(y, "max"), n_players=(x, "count"))
    out["length"] = out.xmax - out.xmin
    out["width"] = out.ymax - out.ymin
    d = df[[*keys, x, y]].join(out[["cx", "cy"]], on=keys)
    d["r"] = np.hypot(d[x] - d.cx, d[y] - d.cy)
    out["spread"] = d.groupby(keys, sort=False)["r"].mean()  # mean distance to centroid
    out["hull_area"] = g[[x, y]].apply(lambda s: hull_area(s.to_numpy()))
    return out.drop(columns=["xmin", "xmax", "ymin", "ymax"]).reset_index()


def add_kinematics(df: pd.DataFrame, track_keys: list, fps: float, smooth: int = 1,
                   x: str = "x", y: str = "y", max_gap: int = 3) -> pd.DataFrame:
    """Add speed (m/s), turn_rate (deg/s) and dt (s) to each row of a player track.

    A track is one player's continuous trajectory (soccer: player in a match;
    football: player in a play). `smooth` is a centred rolling mean in frames.
    """
    df = df.sort_values([*track_keys, "frame"]).reset_index(drop=True)
    if smooth > 1:
        n = len(track_keys)
        for c, new in ((x, "_xs"), (y, "_ys")):
            r = df.groupby(track_keys, sort=False)[c].rolling(smooth, center=True, min_periods=1).mean()
            df[new] = r.droplevel(list(range(n)))
    else:
        df["_xs"], df["_ys"] = df[x], df[y]

    g = df.groupby(track_keys, sort=False)
    dframe = g["frame"].diff()
    ok = dframe.between(1, max_gap)
    dt = dframe / fps
    vx = g["_xs"].diff() / dt
    vy = g["_ys"].diff() / dt
    df["dt"] = dt.where(ok)
    df["speed"] = np.hypot(vx, vy).where(ok)

    df["_h"] = np.arctan2(vy, vx)
    dh = df.groupby(track_keys, sort=False)["_h"].diff()
    dh = (dh + np.pi) % (2 * np.pi) - np.pi
    prev_speed = df.groupby(track_keys, sort=False)["speed"].shift(1)
    turn = np.degrees(np.abs(dh)) / dt
    valid = ok & (df.speed >= MIN_TURN_SPEED) & (prev_speed >= MIN_TURN_SPEED)
    df["turn_rate"] = turn.where(valid)
    return df.drop(columns=["_xs", "_ys", "_h"])


def sprint_summary(df: pd.DataFrame, player_key: str, track_keys: list, fps: float) -> pd.DataFrame:
    """Per-player running profile over live play only (rows where df.live is True)."""
    d = df[df["live"]].copy()
    d["step"] = d["speed"].fillna(0) / fps  # metres covered in this frame
    vmax = d.groupby(player_key)["speed"].quantile(0.99).rename("vmax")
    d = d.join(vmax, on=player_key)
    sprinting = d.speed > SPRINT_MS
    prev = sprinting.groupby([d[k] for k in track_keys]).shift(1, fill_value=False)
    d["sprint_start"] = sprinting & ~prev
    d["hsr_m"] = d.step.where(d.speed > HSR_MS, 0)
    d["sprint_m"] = d.step.where(sprinting, 0)
    d["rel_sprint_m"] = d.step.where(d.speed > REL_SPRINT * d.vmax, 0)
    out = d.groupby(player_key).agg(live_min=("step", "size"), dist_m=("step", "sum"),
                                     hsr_m=("hsr_m", "sum"), sprint_m=("sprint_m", "sum"),
                                     rel_sprint_m=("rel_sprint_m", "sum"),
                                     n_sprints=("sprint_start", "sum"), vmax=("vmax", "first"))
    out["live_min"] = out.live_min / fps / 60
    return out.reset_index().rename(columns={player_key: "player"})


def carrier_profile(df, player_key, min_frames=30):
    """Speed with vs without the object, per player, plus carrier frames for turn analysis."""
    d = df[df["live"] & df.speed.notna()]
    vmax = d.groupby(player_key)["speed"].quantile(0.99).rename("vmax")
    per = d.groupby([player_key, "is_carrier"])["speed"].agg(p95=lambda s: s.quantile(0.95), n="size").unstack()
    per = per.reindex(columns=pd.MultiIndex.from_product([["p95", "n"], [False, True]]))
    per.columns = [f"{a}_{'with' if b else 'without'}" for a, b in per.columns]
    per = per.dropna()
    per = per[per.n_with >= min_frames].join(vmax).reset_index().rename(columns={player_key: "player"})
    per["with_vs_without"] = per.p95_with / per.p95_without

    c = d[d.is_carrier].join(vmax, on=player_key)
    c = c.assign(speed_pct=c.speed / c.vmax)[["speed", "speed_pct", "turn_rate"]]
    return per, c


## 3. Soccer pipeline (SkillCorner)
Possession state comes from SkillCorner's per-frame possession. Shape uses the 10 outfield players. Dynamic events give carry vs pass progression, EPV and how each team possession ends.

In [15]:
FPS = 10


def _read_tracking(path: Path) -> pd.DataFrame:
    with open(path) as f:
        head = f.read(200)
    if head.startswith("version https://git-lfs"):
        raise RuntimeError(f"{path.name} is a Git LFS pointer. Run `git lfs pull` in the repo first.")
    rows = []
    with open(path) as f:
        for line in f:
            fr = json.loads(line)
            if fr.get("period") is None:
                continue
            poss = fr.get("possession") or {}
            for p in fr.get("player_data") or []:
                rows.append((fr["frame"], fr["period"], p["player_id"], p["x"], p["y"],
                             bool(p.get("is_detected")), poss.get("group"), poss.get("player_id")))
    cols = ["frame", "period", "player_id", "x", "y", "is_detected", "poss_group", "poss_player"]
    df = pd.DataFrame(rows, columns=cols)
    df[["x", "y"]] = df[["x", "y"]].astype(float)
    return df


def load_match(match_dir: Path):
    match_dir = Path(match_dir)
    mid = match_dir.name
    meta = json.loads((match_dir / f"{mid}_match.json").read_text())
    players = pd.DataFrame([{
        "player_id": p["id"], "team_id": p["team_id"],
        "role": (p.get("player_role") or {}).get("acronym"),
    } for p in meta["players"]])
    home, away = meta["home_team"]["id"], meta["away_team"]["id"]
    L, W = meta["pitch_length"], meta["pitch_width"]

    tr = _read_tracking(match_dir / f"{mid}_tracking_extrapolated.jsonl").merge(players, on="player_id")
    tr["poss_team"] = tr.poss_group.map({"home team": home, "away team": away})
    tr["state"] = np.where(tr.poss_team.isna(), None, np.where(tr.team_id == tr.poss_team, "in", "out"))
    tr["live"] = tr.poss_team.notna()  # proxy for live play: someone has the ball

    # Attacking direction per team and period -> x of each team's own goal.
    sides = meta.get("home_team_side") or []
    def home_sign(period):
        if period - 1 < len(sides):
            return 1 if sides[period - 1] == "left_to_right" else -1
        gk = tr[(tr.period == period) & (tr.team_id == home) & (tr.role == "GK")]
        return 1 if gk.x.mean() < 0 else -1
    sign = {(p, home): home_sign(p) for p in tr.period.unique()}
    sign.update({(p, away): -s for (p, _), s in list(sign.items())})
    tr["attack_sign"] = [sign[k] for k in zip(tr.period, tr.team_id)]
    tr["own_goal_x"] = -tr.attack_sign * L / 2

    events = pd.read_csv(match_dir / f"{mid}_dynamic_events.csv", low_memory=False)
    return tr, events, dict(match_id=int(mid), L=L, W=W, home=home, away=away)


def _transitions(tr, info, before_s=5, after_s=10, hold_s=2):
    """Team shape around every open-play possession change."""
    poss = tr.groupby("frame").poss_team.first().ffill(limit=FPS)
    period = tr.groupby("frame").period.first()
    prev = poss.shift()
    chg = poss.ne(prev) & poss.notna() & prev.notna()
    frames, vals = poss.index.to_numpy(), poss.to_numpy()
    starts = []
    for i in np.flatnonzero(chg.to_numpy()):
        held = vals[i:i + hold_s * FPS]
        if len(held) == hold_s * FPS and (held == vals[i]).all():
            starts.append((frames[i], vals[i], period.iloc[i]))
    if not starts:
        return pd.DataFrame()

    outfield = tr[tr.role != "GK"]
    windows = []
    for t0, winner, per in starts:
        w = outfield[(outfield.frame.between(t0 - before_s * FPS, t0 + after_s * FPS)) & (outfield.period == per)]
        if w.empty:
            continue
        s = team_shapes(w, ["frame", "team_id"])
        s["rel_t"] = (s.frame - t0) / FPS
        s["role"] = np.where(s.team_id == winner, "won ball", "lost ball")
        s["transition_frame"] = t0
        windows.append(s)
    out = pd.concat(windows, ignore_index=True)
    out["match_id"] = info["match_id"]
    return out


def _progression(events, info):
    """Carry vs pass progression and value per player possession; outcome of each team possession."""
    pp = events[events.event_type == "player_possession"].sort_values("frame_start").copy()
    sign = np.where(pp.attacking_side == "left_to_right", 1, -1)
    pp["carry_m"] = (pp.x_end - pp.x_start) * sign
    nxt = pp.shift(-1)
    completed = (pp.end_type == "pass") & (nxt.team_id == pp.team_id)
    pp["pass_m"] = np.where(completed, (nxt.x_start - pp.x_end) * sign, np.nan)
    pp["epv_delta"] = pp.get("possession_epv_delta_for", np.nan)

    last = pp[pp.last_player_possession_in_team_possession.astype(str) == "True"].copy()
    last["outcome"] = np.select(
        [last.end_type == "shot", last.game_interruption_after.notna()],
        ["shot", "stoppage"], "lost in open play")
    pp["match_id"] = last["match_id"] = info["match_id"]
    return (pp[["match_id", "team_id", "carry_m", "pass_m", "end_type", "epv_delta"]],
            last[["match_id", "team_id", "outcome"]])


def process_match(match_dir, stride: int = 2) -> dict:
    tr, events, info = load_match(match_dir)
    tr = add_kinematics(tr, ["player_id"], fps=FPS, smooth=5)
    tr["is_carrier"] = tr.player_id == tr.poss_player

    outfield = tr[(tr.role != "GK") & tr.live & (tr.frame % stride == 0)]
    shapes = team_shapes(outfield, ["frame", "period", "team_id", "state", "own_goal_x"])
    shapes["dist_own_goal"] = (shapes.cx - shapes.own_goal_x).abs()
    shapes["field_area"] = info["L"] * info["W"]

    carriers, carrier_frames = carrier_profile(tr, "player_id")
    sprints = sprint_summary(tr, "player_id", ["player_id"], fps=FPS)
    progression, possessions = _progression(events, info)

    out = dict(shapes=shapes, transitions=_transitions(tr, info), carriers=carriers,
               carrier_frames=carrier_frames, sprints=sprints,
               progression=progression, possessions=possessions)
    for df in out.values():
        if not df.empty:
            df["sport"], df["match_id"] = "soccer", info["match_id"]
    return out


def soccer_process_all(opendata_dir, max_matches=None, stride=2) -> dict:
    match_dirs = sorted(p for p in (Path(opendata_dir) / "data" / "matches").iterdir() if p.is_dir())
    results = []
    for md in match_dirs[:max_matches]:
        print(f"[soccer] {md.name}")
        results.append(process_match(md, stride=stride))
    return {k: pd.concat([r[k] for r in results], ignore_index=True) for k in results[0]}


## 4. Football pipeline (StatsBomb + nflverse)
StatsBomb's 30 fps tracking is downsampled to 10 fps to match SkillCorner, and both are smoothed the same way. Coordinates are flipped so the offence always attacks +x, then converted to metres. The offence is "in possession" from the snap to the end of the play. The ball carrier comes from StatsBomb's events (excluding QBs in the pocket). Frames with more than one player off camera are dropped from the shape analysis.

In [ ]:
FPS = 10                      # we downsample StatsBomb's 30 fps to match SkillCorner
YD = 0.9144
FIELD_L, FIELD_W = 120 * YD, 53.3 * YD   # including end zones
OFF_GOAL_X, DEF_GOAL_X = 0.0, 100 * YD   # goal lines after normalising (offence attacks +x)
MIN_PLAYERS = 10              # drop frames where too many players are off camera


def _read_game(path: Path) -> tuple:
    g = json.load(gzip.open(path))
    rows, plays = [], []
    for p in g["plays"]:
        off, ltr = p["play_offense_team_id"], p["offense_left_to_right"]
        plays.append(dict(game_id=g["game_id"], old_game_id=str(g.get("nfl_old_game_id")),
                          play_uuid=p["play_uuid"], play_type=p["play_type"],
                          season=g["season_name"]))
        for t in p["tracks"]:
            pl = t.get("player") or {}
            side = "off" if t["team_id"] == off else "def"
            last = None
            for s in t["steps"]:
                fr = int(s["timestamp"] // 100_000)  # 0.1 s bins -> 10 Hz, same clock as the events
                if fr == last:
                    continue
                last = fr
                rows.append((p["play_uuid"], pl.get("player_id"), pl.get("position_code"), side,
                             fr, s["time_since_snap"], s["x"], s["y"], ltr))
    df = pd.DataFrame(rows, columns=["play_uuid", "player_id", "position", "side", "frame",
                                     "time_since_snap", "x", "y", "ltr"])
    df[["x", "y"]] = df[["x", "y"]].astype(float)
    flip = ~df.ltr.astype(bool)                       # normalise: offence always attacks +x
    df.loc[flip, "x"] = 100 - df.loc[flip, "x"]
    df.loc[flip, "y"] = 53.3 - df.loc[flip, "y"]
    df[["x", "y"]] *= YD
    df["game_id"] = g["game_id"]
    df["live"] = df.time_since_snap >= 0              # tracking ends at the end of the play
    df["state"] = df.side.map({"off": "in", "def": "out"})
    df["own_goal_x"] = df.side.map({"off": OFF_GOAL_X, "def": DEF_GOAL_X})
    return df.drop(columns="ltr"), pd.DataFrame(plays)


def load_carriers(sb_dir) -> pd.DataFrame:
    """Ball carrier every 0.1 s, from StatsBomb event data (seasons 2017-2022)."""
    cols = ["play_uuid", "event_video_timestamp", "event_ball_carrier"]
    files = sorted((Path(sb_dir) / "data" / "events").glob("*.csv"))
    ev = pd.concat([pd.read_csv(f, usecols=cols) for f in files], ignore_index=True)
    ev = ev.dropna(subset=["event_ball_carrier"])
    ev["frame"] = np.floor(ev.event_video_timestamp * 10).astype(int)
    ev = ev.rename(columns={"event_ball_carrier": "carrier_id"})
    return ev.drop_duplicates(["play_uuid", "frame"])[["play_uuid", "frame", "carrier_id"]]


def _mark_carriers(df, carriers):
    """Carrier = the player StatsBomb tags as holding the ball, excluding QBs in the pocket."""
    c = carriers[carriers.play_uuid.isin(df.play_uuid.unique())].sort_values("frame")
    d = df[["play_uuid", "frame"]].drop_duplicates().sort_values("frame")
    d = pd.merge_asof(d, c, on="frame", by="play_uuid", tolerance=2, direction="backward")
    df = df.merge(d, on=["play_uuid", "frame"], how="left")
    df["is_carrier"] = (df.player_id == df.carrier_id) & (df.position != "QB") & df.live
    return df.drop(columns="carrier_id")


def process_game(path, carriers, stride=2) -> dict:
    df, plays = _read_game(path)
    df = _mark_carriers(df, carriers)
    df = df.dropna(subset=["player_id"])
    df = add_kinematics(df, ["play_uuid", "player_id"], fps=FPS, smooth=5)

    live = df[df.live & (df.frame % stride == 0)]
    shapes = team_shapes(live, ["play_uuid", "frame", "side", "state", "own_goal_x"])
    shapes = shapes[shapes.n_players >= MIN_PLAYERS]
    shapes["dist_own_goal"] = (shapes.cx - shapes.own_goal_x).abs()
    shapes["field_area"] = FIELD_L * FIELD_W
    shapes["game_id"] = plays.game_id.iloc[0]

    # Keep a compact copy of live frames; speed profiles are computed across all games at the end
    frames = df.loc[df.live, ["game_id", "play_uuid", "player_id", "frame", "speed", "turn_rate", "is_carrier"]]
    frames = frames.astype({"speed": "float32", "turn_rate": "float32"})
    return dict(shapes=shapes, frames=frames, plays=plays)


def football_process_all(sb_dir, max_games=None, stride=2):
    sb_dir = Path(sb_dir)
    carriers = load_carriers(sb_dir)
    paths = sorted((sb_dir / "data" / "tracking").glob("*.json.gz"))
    if max_games:  # spread the sample across seasons (2016 games have no carrier data)
        paths = paths[::max(1, len(paths) // max_games)][:max_games]
    res = []
    for path in paths:
        print(f"[football] {path.name}")
        res.append(process_game(path, carriers, stride))
    shapes = pd.concat([r["shapes"] for r in res], ignore_index=True)
    frames = pd.concat([r["frames"] for r in res], ignore_index=True)
    plays = pd.concat([r["plays"] for r in res], ignore_index=True)

    frames["live"] = True
    carriers_out, carrier_frames = carrier_profile(frames, "player_id")
    sprints = sprint_summary(frames, "player_id", ["play_uuid", "player_id"], fps=FPS)
    out = dict(shapes=shapes, carriers=carriers_out, carrier_frames=carrier_frames,
               sprints=sprints, plays=plays)
    for df in out.values():
        df["sport"] = "football"
    return out


# ---- Play-by-play (nflverse): turnovers, EPA, air vs ground progression ----

def load_pbp(seasons, old_game_ids=None) -> pd.DataFrame:
    """nflverse play-by-play, optionally restricted to the games in the tracking sample."""
    url = "https://github.com/nflverse/nflverse-data/releases/download/pbp/play_by_play_{}.parquet"
    pbp = pd.concat([pd.read_parquet(url.format(s)) for s in seasons], ignore_index=True)
    if old_game_ids is not None:
        pbp = pbp[pbp.old_game_id.astype(str).isin(set(old_game_ids))]
    return pbp


def pbp_summary(pbp: pd.DataFrame) -> dict:
    p = pbp[pbp.play_type.isin(["pass", "run"])].copy()
    p["turnover"] = (p.interception.fillna(0) + p.fumble_lost.fillna(0)) > 0
    drives = p.groupby(["game_id", "posteam", "drive"]).agg(plays=("play_id", "size"), turnover=("turnover", "max"))
    done = p.complete_pass == 1
    air = p.loc[done, "air_yards"].clip(lower=0).sum()
    ground = p.loc[done, "yards_after_catch"].clip(lower=0).sum() + p.loc[p.play_type == "run", "yards_gained"].clip(lower=0).sum()
    return dict(
        epa=p[["play_type", "epa"]].assign(sport="football"),
        rates=dict(turnovers_per_play=p.turnover.mean(), turnovers_per_drive=drives.turnover.mean(),
                   plays_per_drive=drives.plays.mean(), share_progression_through_air=air / (air + ground)),
    )


## 5. Figures and tables

In [17]:
COLORS = {"soccer": "#2a9d8f", "football": "#e76f51"}
STATE = {"in": "In possession", "out": "Out of possession"}


def _save(fig, out, name):
    fig.tight_layout()
    fig.savefig(out / name, dpi=200)
    plt.show()


def shape_by_possession(shapes, out):
    """H1: is the gap between attacking and defending shape bigger in soccer?"""
    s = shapes.assign(hull_pct=100 * shapes.hull_area / shapes.field_area)
    metrics = [("width", "Team width (m)"), ("hull_pct", "Convex hull (% of field)"),
               ("dist_own_goal", "Centroid distance from own goal (m)")]
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, (col, label) in zip(axes, metrics):
        data, pos, cols = [], [], []
        for i, sport in enumerate(["soccer", "football"]):
            for j, state in enumerate(["in", "out"]):
                v = s[(s.sport == sport) & (s.state == state)][col].dropna()
                if len(v):
                    data.append(v); pos.append(i * 3 + j); cols.append(COLORS[sport])
        bp = ax.boxplot(data, positions=pos, widths=0.8, showfliers=False, patch_artist=True)
        for patch, c, p in zip(bp["boxes"], cols, pos):
            patch.set_facecolor(c); patch.set_alpha(1.0 if p % 3 == 0 else 0.45)
        ax.set_xticks([0.5, 3.5], ["Soccer", "Football"]); ax.set_title(label, fontsize=10)
    fig.suptitle("Team shape in possession (solid) vs out of possession (faded)")
    _save(fig, out, "fig1_shape_by_possession.png")

    t = s.groupby(["sport", "state"])[["width", "length", "hull_pct", "spread", "dist_own_goal"]].median().unstack("state")
    for col in ["width", "hull_pct", "spread"]:
        t[(col, "in/out ratio")] = t[(col, "in")] / t[(col, "out")]
    return t.round(2)


def transitions(trans, out):
    """H2: how fast does a soccer team reshape after the ball changes hands?"""
    if trans.empty:
        return pd.DataFrame()
    m = trans.groupby(["role", "rel_t"]).hull_area.agg(["mean", "sem"]).reset_index()
    fig, ax = plt.subplots(figsize=(7, 4))
    for role, c in [("won ball", COLORS["soccer"]), ("lost ball", "#264653")]:
        d = m[m.role == role]
        ax.plot(d.rel_t, d["mean"], color=c, label=f"Team that {role}")
        ax.fill_between(d.rel_t, d["mean"] - 1.96 * d["sem"], d["mean"] + 1.96 * d["sem"], color=c, alpha=0.2)
    ax.axvline(0, color="grey", ls="--", lw=1)
    ax.set_xlabel("Seconds from possession change"); ax.set_ylabel("Outfield convex hull (m²)")
    ax.set_title("Soccer: expanding and contracting after a turnover"); ax.legend()
    _save(fig, out, "fig2_transitions.png")
    return m


def carrying(carriers, carrier_frames, out):
    """H3: how much does carrying the object cost, and how much speed survives a turn?"""
    bins = [0, 30, 60, 90, 135, 180, 270, 360]
    cf = carrier_frames.dropna(subset=["turn_rate", "speed_pct"]).copy()
    cf["turn_bin"] = pd.cut(cf.turn_rate.clip(upper=359), bins, right=False)
    turn = cf.groupby(["sport", "turn_bin"], observed=True).speed_pct.median().unstack("sport")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for sport in [c for c in ["soccer", "football"] if c in turn]:
        axes[0].plot(range(len(turn)), 100 * turn[sport], marker="o", color=COLORS[sport], label=sport.title())
        v = carriers[carriers.sport == sport].with_vs_without.dropna()
        axes[1].hist(100 * v, bins=30, alpha=0.6, color=COLORS[sport], label=f"{sport.title()} (median {100 * v.median():.0f}%)")
    axes[0].set_xticks(range(len(turn)), [str(b) for b in turn.index], rotation=30)
    axes[0].set_xlabel("Turn rate while carrying (deg/s)"); axes[0].set_ylabel("Median speed (% of own top speed)")
    axes[0].set_title("Speed kept while turning with the object"); axes[0].legend()
    axes[1].set_xlabel("Top carrying speed as % of top speed without the object")
    axes[1].set_ylabel("Players"); axes[1].set_title("The cost of carrying"); axes[1].legend()
    _save(fig, out, "fig3_carrying.png")
    return turn.round(3)


def sprinting(sprints, out):
    """H5: does stop-start play let football players sprint harder per live minute?"""
    g = sprints[sprints.live_min > 0].groupby("sport")[["live_min", "dist_m", "hsr_m", "sprint_m", "rel_sprint_m", "n_sprints"]].sum()
    per_min = g.drop(columns="live_min").div(g.live_min, axis=0)
    per_min["sprint_share_of_distance"] = g.sprint_m / g.dist_m
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
    for ax, (col, label) in zip(axes, [("dist_m", "Distance per live minute (m)"),
                                        ("sprint_share_of_distance", "Share of distance above 7 m/s"),
                                        ("rel_sprint_m", "Metres above 85% of own top speed, per live min")]):
        ax.bar(per_min.index.str.title(), per_min[col], color=[COLORS[s] for s in per_min.index])
        ax.set_title(label, fontsize=10)
    _save(fig, out, "fig4_sprinting.png")
    return per_min.round(3)


def progression(soccer_prog, soccer_poss, pbp_rates, out):
    """H4: how progression is split, how often the object is lost, and value per moment."""
    carry = soccer_prog.carry_m.clip(lower=0).sum()
    passed = soccer_prog.pass_m.clip(lower=0).sum()
    outcomes = soccer_poss.outcome.value_counts(normalize=True)
    rows = {"soccer": dict(share_progression_through_air=passed / (passed + carry),
                           turnovers_per_possession=outcomes.get("lost in open play", np.nan),
                           player_possessions_per_team_possession=len(soccer_prog) / max(len(soccer_poss), 1))}
    if pbp_rates:
        rows["football"] = dict(share_progression_through_air=pbp_rates["share_progression_through_air"],
                                turnovers_per_possession=pbp_rates["turnovers_per_drive"],
                                player_possessions_per_team_possession=pbp_rates["plays_per_drive"])
    t = pd.DataFrame(rows).T
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    for ax, (col, label) in zip(axes, [("share_progression_through_air", "Share of forward progress by pass"),
                                        ("turnovers_per_possession", "Possessions lost in play\n(soccer: team possession, football: drive)")]):
        ax.bar(t.index.str.title(), t[col], color=[COLORS[s] for s in t.index]); ax.set_title(label, fontsize=10)
    _save(fig, out, "fig5_progression_turnovers.png")
    return t.round(3)


## 6. Run the pipelines
This is the slow part. Try the sample limits first.

In [18]:
sc = soccer_process_all(SKILLCORNER_DIR, MAX_MATCHES, stride=SOCCER_STRIDE)
fb = football_process_all(STATSBOMB_DIR, MAX_GAMES, stride=FOOTBALL_STRIDE)
seasons = sorted({int(str(s)[:4]) for s in fb["plays"].season})
try:
    pbp = pbp_summary(load_pbp(seasons, fb["plays"].old_game_id.unique()))
except Exception as e:  # no internet, missing pyarrow, etc.
    print(f"[pbp] skipped: {e}")
    pbp = None

{k: len(v) for k, v in {**{f"soccer_{k}": v for k, v in sc.items()}, **{f"football_{k}": v for k, v in fb.items()}}.items()}

[soccer] 1874553
[soccer] 1886347
[football] SB_tracking_20160911_NE_ARI.json.gz
[football] SB_tracking_20160918_MIA_NE.json.gz
[football] SB_tracking_20160922_HOU_NE.json.gz
[football] SB_tracking_20161002_BUF_NE.json.gz


AttributeError: 'DataFrame' object has no attribute 'n_with'

In [ ]:
# Save raw metrics so you don't have to rerun the pipelines
for name, df in {**{f"soccer_{k}": v for k, v in sc.items()},
                 **{f"football_{k}": v for k, v in fb.items()}}.items():
    df.to_parquet(OUT / f"{name}.parquet")

## H1: Team shape in vs out of possession
If the central goal pulls soccer defences narrow, the in/out ratio for width and hull area should be clearly above 1 for soccer and closer to 1 for football. Hull area is shown as a share of the field so the different field sizes don't drive the result.

In [ ]:
h1 = shape_by_possession(pd.concat([sc["shapes"], fb["shapes"]]), OUT)
h1

## H2: Transitions in soccer
Average outfield hull area from 5 s before to 10 s after an open-play possession change. Football has no direct equivalent, since teams reset between plays.

In [ ]:
h2 = transitions(sc["transitions"], OUT)

## H3: Carrying and turning
Left: how much of their own top speed carriers keep at each turn rate. Right: each player's top carrying speed as a share of their top speed without the object.

In [ ]:
h3 = carrying(pd.concat([sc["carriers"], fb["carriers"]]),
              pd.concat([sc["carrier_frames"], fb["carrier_frames"]]), OUT)
h3

## H4: Progression and turnovers
Soccer: share of forward metres gained by pass vs carry, and the share of team possessions lost in open play. Football (nflverse, same games as the tracking): share of yards gained through the air (air yards on completions) vs on the ground, and the share of drives ending in an interception or lost fumble.

In [ ]:
h4 = progression(sc["progression"], sc["possessions"], pbp["rates"] if pbp else None, OUT)
h4

### Value per moment (EPV vs EPA)
These are on different scales (goals vs points), so compare shapes of the distributions rather than raw values.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
sc["progression"].epv_delta.dropna().clip(-0.1, 0.1).hist(bins=60, ax=axes[0], color=COLORS["soccer"])
axes[0].set_title("Soccer: EPV change per player possession")
if pbp:
    pbp["epa"].epa.dropna().clip(-6, 6).hist(bins=60, ax=axes[1], color=COLORS["football"])
axes[1].set_title("Football: EPA per play")
plt.tight_layout(); plt.show()

## H5: Sprinting and rest
Distance and sprinting per minute of live play. If rest between plays matters, football should show less distance per live minute but a larger share of it at sprint speed.

In [ ]:
h5 = sprinting(pd.concat([sc["sprints"], fb["sprints"]]), OUT)
h5

## Summary tables

In [ ]:
with open(OUT / "summary.md", "w") as f:
    for title, t in {"H1 team shape by possession": h1, "H3 carrying and turning": h3,
                     "H4 progression and turnovers": h4, "H5 sprinting": h5}.items():
        f.write(f"## {title}\n\n{t.to_markdown()}\n\n")
print(f"Saved figures and tables to {OUT.resolve()}")

## Definitions and caveats
- **Live play (soccer):** frames where a team has possession. A proxy, not an official ball-in-play flag.
- **Top speed:** each player's 99th percentile speed within the match (soccer) or tracking week (football).
- **Soccer turnover:** a team possession that ends without a shot or a stoppage. **Football turnover:** interception or lost fumble, per drive.
- **Broadcast tracking:** SkillCorner extrapolates off-screen players, so treat speed and turn metrics with care; filter on `is_detected` for a stricter version.
- **Comparison units differ:** soccer possession flips live; football possession is fixed for a play. State this up front in the paper.
- **Football sample:** only games involving Tom Brady's teams (NE 2016-19, TB 2020-22), so one side is always the same franchise. Both offences are tracked, which helps. Ball carriers come from events, which start in 2017, so 2016 games add shape and sprint data but no carries.
- **Both sports are video-derived** (SkillCorner from broadcast, StatsBomb from video at 30 fps), so tracking error is of a similar kind on each side, a useful point for the methods section.